In [5]:
# Cell 1
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

print("✅ Cell 1 Complete: All libraries imported.")

✅ Cell 1 Complete: All libraries imported.


In [6]:
# Cell 2
df = pd.read_csv('data/dataset.csv')
raw_count = len(df)

# Standardize column headers
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# Process Target
df = df.dropna(subset=['fraud_reported'])
df['fraud_reported'] = df['fraud_reported'].str.strip().map({'Y': 1, 'N': 0})

# Clean Dirty Symbols ('*')
dirty_cols = ['marital_status', 'witness_present', 'age_of_vehicle', 'injury_claim']
for col in dirty_cols:
    df[col] = pd.to_numeric(df[col].astype(str).replace('*', np.nan), errors='coerce')
    df[col] = df[col].fillna(df[col].median())

# Select Feature Columns
num_cols = [
    'age_of_driver', 'marital_status', 'safety_rating', 'annual_income',
    'high_education', 'address_change', 'past_num_of_claims', 'witness_present',
    'liab_prct', 'police_report', 'age_of_vehicle', 'vehicle_price',
    'total_claim', 'injury_claim', 'policy_deductible', 'annual_premium',
    'days_open', 'form_defects'
]
cat_cols = ['gender', 'property_status', 'accident_site', 'channel', 'vehicle_category']

X = df[num_cols + cat_cols]
y = df['fraud_reported']

print(f"✅ Cell 2 Complete: Dataset cleaned successfully. {len(X)} rows ready for training.")

✅ Cell 2 Complete: Dataset cleaned successfully. 11994 rows ready for training.


In [7]:
# Cell 3
class LogisticRegressionScratch:
    def __init__(self, lr=0.01, n_iters=500):
        self.lr = lr
        self.n_iters = n_iters
        self.weights = None
        self.bias = None

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in range(self.n_iters):
            linear_model = np.dot(X, self.weights) + self.bias
            y_predicted = self._sigmoid(linear_model)
            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / n_samples) * np.sum(y_predicted - y)
            self.weights -= self.lr * dw
            self.bias -= self.lr * db

    def predict(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        y_pred = self._sigmoid(linear_model)
        return np.array([1 if i >= 0.5 else 0 for i in y_pred])

print("✅ Cell 3 Complete: Custom Logistic Regression Scratch Model Defined.")

✅ Cell 3 Complete: Custom Logistic Regression Scratch Model Defined.


In [8]:
# Cell 4
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
])

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

# Scratch Model Execution
scratch_model = LogisticRegressionScratch(lr=0.05, n_iters=500)
scratch_model.fit(X_train_trans, y_train.values)
scratch_preds = scratch_model.predict(X_test_trans)

# Library Execution
lib_lr = LogisticRegression(max_iter=500)
lib_lr.fit(X_train_trans, y_train)
lib_preds = lib_lr.predict(X_test_trans)

print(f"✅ Cell 4 Complete Benchmark Results:")
print(f"   Scratch Model Accuracy: {accuracy_score(y_test, scratch_preds)*100:.2f}%")
print(f"   Library Model Accuracy: {accuracy_score(y_test, lib_preds)*100:.2f}%")

✅ Cell 4 Complete Benchmark Results:
   Scratch Model Accuracy: 77.57%
   Library Model Accuracy: 77.24%


In [13]:
# Cell 5
best_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42))
])

best_pipeline.fit(X_train, y_train)

y_pred = best_pipeline.predict(X_test)
y_proba = best_pipeline.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_proba)

print(f"✅ Cell 5 Complete Best Model Metrics:")
print(f"   Accuracy : {acc*100:.2f}%")
print(f"   F1-Score : {f1*100:.2f}%")
print(f"   ROC-AUC  : {roc*100:.2f}%")

✅ Cell 5 Complete Best Model Metrics:
   Accuracy : 77.62%
   F1-Score : 19.00%
   ROC-AUC  : 64.35%


In [14]:
# Cell 6: Force Export Pickle and app.py
model_payload = {
    'pipeline': best_pipeline,
    'num_cols': num_cols,
    'cat_cols': cat_cols,
    'metrics': {'accuracy': f"{acc * 100:.1f}%", 'f1_score': f"{f1 * 100:.1f}%", 'roc_auc': f"{roc * 100:.1f}%"},
    'eda_summary': {'raw_records': raw_count, 'rows_removed': raw_count - len(df), 'final_records': len(df)}
}

# 1. Export pkl
with open('insurance_fraud_model.pkl', 'wb') as f:
    pickle.dump(model_payload, f)

# 2. Export app.py
app_code = """from flask import Flask, render_template_string, request
import pickle
import pandas as pd

app = Flask(__name__)

with open('insurance_fraud_model.pkl', 'rb') as f:
    payload = pickle.load(f)

pipeline = payload['pipeline']
metrics = payload['metrics']

@app.route('/')
def home():
    return f"<h1>Vehicle Insurance Fraud Portal</h1><p>Model Accuracy: {metrics['accuracy']}</p>"

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=True)
"""

with open('app.py', 'w') as f:
    f.write(app_code)

print("✅ Cell 6 Complete: File Creation Verified!")
import os
print("   'insurance_fraud_model.pkl' created:", os.path.exists('insurance_fraud_model.pkl'))
print("   'app.py' created:", os.path.exists('app.py'))

✅ Cell 6 Complete: File Creation Verified!
   'insurance_fraud_model.pkl' created: True
   'app.py' created: True


In [16]:
import pickle

# Check if model pipeline exists
if 'best_pipeline' in globals():

    # Optional EDA values
    raw_records = globals().get('raw_count', 0)
    final_records = globals().get('final_count', 0)

    if raw_records and final_records:
        rows_removed = raw_records - final_records
    else:
        rows_removed = 0

    model_payload = {
        'pipeline': best_pipeline,
        'num_cols': num_cols,
        'cat_cols': cat_cols,

        'metrics': {
            'accuracy': f"{acc * 100:.1f}%",
            'f1_score': f"{f1 * 100:.1f}%",
            'roc_auc': f"{roc * 100:.1f}%"
        },

        'eda_summary': {
            'raw_records': raw_records,
            'rows_removed': rows_removed,
            'final_records': final_records
        }
    }

    with open('insurance_fraud_model.pkl', 'wb') as f:
        pickle.dump(model_payload, f)

    print("SUCCESS: insurance_fraud_model.pkl created successfully!")

else:
    print("ERROR: best_pipeline not found.")
    print("Run the model training cell first.")

SUCCESS: insurance_fraud_model.pkl created successfully!


In [17]:
# Cell 8: Streamlit app.py with all exact features & About Us tab
streamlit_app_code = '''import streamlit as st
import pickle
import pandas as pd
import numpy as np

# Page configuration
st.set_page_config(
    page_title="FraudGuard AI - Insurance Fraud Detection",
    page_icon="🛡️",
    layout="wide"
)

# Load trained model bundle
@st.cache_resource
def load_payload():
    with open('insurance_fraud_model.pkl', 'rb') as f:
        return pickle.load(f)

payload = load_payload()
pipeline = payload['pipeline']
metrics = payload['metrics']
eda_info = payload['eda_summary']

st.title("🛡️ FraudGuard AI: Vehicle Insurance Fraud Portal")
st.markdown("Assess claim fraud risks, inspect model evaluation metrics, explore dataset summary statistics, and view author information.")

tabs = st.tabs(["📋 Claim Fraud Prediction", "📊 Model Performance", "📈 Data Insights", "👤 About Us"])

# TAB 1: Complete Prediction Form (All 23 Input Features)
with tabs[0]:
    st.header("Assess New Claim")
    st.write("Fill in the 23 claim features below to predict the likelihood of fraud:")

    with st.form("fraud_form"):
        col1, col2, col3 = st.columns(3)

        with col1:
            st.subheader("Driver & Policy Info")
            age_of_driver = st.number_input("Age of Driver", min_value=18, max_value=100, value=35)
            gender = st.selectbox("Gender", ["M", "F"])
            marital_status = st.selectbox("Marital Status", [1, 0], format_func=lambda x: "Married" if x == 1 else "Single / Other")
            safety_rating = st.slider("Safety Rating", 0, 100, 75)
            annual_income = st.number_input("Annual Income ($)", min_value=0.0, value=50000.0, step=1000.0)
            high_education = st.selectbox("High Education Completed", [1, 0], format_func=lambda x: "Yes" if x == 1 else "No")
            address_change = st.selectbox("Address Change Recently", [0, 1], format_func=lambda x: "No" if x == 0 else "Yes")
            property_status = st.selectbox("Property Status", ["Own", "Rent"])

        with col2:
            st.subheader("Accident & Claim Details")
            accident_site = st.selectbox("Accident Site", ["Highway", "Local", "Parking Lot"])
            past_num_of_claims = st.number_input("Past Number of Claims", min_value=0, max_value=20, value=0)
            witness_present = st.selectbox("Witness Present", [0, 1], format_func=lambda x: "No" if x == 0 else "Yes")
            liab_prct = st.slider("Liability Percentage (%)", 0, 100, 25)
            channel = st.selectbox("Policy Channel", ["Phone", "Online", "Broker"])
            police_report = st.selectbox("Police Report Filed", [0, 1], format_func=lambda x: "No" if x == 0 else "Yes")
            days_open = st.number_input("Days Claim Open", min_value=0.0, value=8.0, step=0.5)
            form_defects = st.number_input("Form Defects Count", min_value=0, max_value=15, value=0)

        with col3:
            st.subheader("Vehicle & Financial Values")
            age_of_vehicle = st.number_input("Age of Vehicle (years)", min_value=0, max_value=30, value=4)
            vehicle_category = st.selectbox("Vehicle Category", ["Compact", "Medium", "Large"])
            vehicle_price = st.number_input("Vehicle Price ($)", min_value=0.0, value=25000.0, step=500.0)
            total_claim = st.number_input("Total Claim Amount ($)", min_value=0.0, value=12000.0, step=500.0)
            injury_claim = st.number_input("Injury Claim Amount ($)", min_value=0.0, value=3000.0, step=500.0)
            policy_deductible = st.selectbox("Policy Deductible ($)", [500, 1000, 2000])
            annual_premium = st.number_input("Annual Premium ($)", min_value=0.0, value=1200.0, step=100.0)

        submit_btn = st.form_submit_button("Evaluate Claim Fraud Risk", use_container_width=True)

    if submit_btn:
        input_data = pd.DataFrame([{
            'age_of_driver': age_of_driver,
            'marital_status': marital_status,
            'safety_rating': safety_rating,
            'annual_income': annual_income,
            'high_education': high_education,
            'address_change': address_change,
            'past_num_of_claims': past_num_of_claims,
            'witness_present': witness_present,
            'liab_prct': liab_prct,
            'police_report': police_report,
            'age_of_vehicle': age_of_vehicle,
            'vehicle_price': vehicle_price,
            'total_claim': total_claim,
            'injury_claim': injury_claim,
            'policy_deductible': policy_deductible,
            'annual_premium': annual_premium,
            'days_open': days_open,
            'form_defects': form_defects,
            'gender': gender,
            'property_status': property_status,
            'accident_site': accident_site,
            'channel': channel,
            'vehicle_category': vehicle_category
        }])

        prediction = pipeline.predict(input_data)[0]
        probability = pipeline.predict_proba(input_data)[0][1]

        st.subheader("Prediction Result")
        if prediction == 1:
            st.error(f"⚠️ **High Fraud Risk Detected** | Confidence Score: **{probability*100:.1f}%**")
        else:
            st.success(f"✅ **Low Fraud Risk / Legitimate Claim** | Confidence Score: **{(1-probability)*100:.1f}%**")

# TAB 2: Model Performance Metrics
with tabs[1]:
    st.header("GradientBoostingClassifier Metrics")
    m1, m2, m3 = st.columns(3)
    m1.metric("Accuracy", metrics['accuracy'])
    m2.metric("F1 Score", metrics['f1_score'])
    m3.metric("ROC AUC", metrics['roc_auc'])

# TAB 3: Dataset Summary
with tabs[2]:
    st.header("Exploratory Data Analysis Summary")
    d1, d2, d3 = st.columns(3)
    d1.metric("Raw Claims Loaded", eda_info['raw_records'])
    d2.metric("Invalid Records Removed", eda_info['rows_removed'])
    d3.metric("Final Training Claims", eda_info['final_records'])

# TAB 4: About Us Section
with tabs[3]:
    st.header("About Us")
    
    st.markdown("""
    ### 👨‍💻 Project Developer Information
    
    **Developer Name:** [Your Name Here]  
    **Role:** Machine Learning Engineer / Data Scientist  
    **Email:** [your.email@example.com]  
    **GitHub / LinkedIn:** [Insert Link]  
    
    ---
    
    ### 🚀 Project Overview
    **FraudGuard AI** is an end-to-end Machine Learning web application built to assist insurance companies in identifying fraudulent vehicle insurance claims.
    
    * **Machine Learning Pipeline:** Implements Scikit-Learn data transformation, handling dirty symbols, scaling, and hyperparameter-tuned classification (`GradientBoostingClassifier`).
    * **Key Capabilities:** Instant claim risk assessment, interactive feature parameter inputs, and real-time confidence scores.
    """)
'''

# Saved using UTF-8 encoding to prevent Windows UnicodeEncodeError
with open('app.py', 'w', encoding='utf-8') as f:
    f.write(streamlit_app_code)

print("✅ Updated 'app.py' created successfully!")

✅ Updated 'app.py' created successfully!
